**CNN and Vision Transformer (ViT) Models**

This notebook defines and tests two models for handwritten digit classification.
- CNNModel: Convolutional Neural Network
- ViTModel: Small Vision Transformer using 2×2 image patches
Expected input shape: [batch_size, 1, 8, 8]

In [ ]:
# libaries

import torch
import torch.nn as nn

2. CNN Model

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

3. Vision Transformer (ViT) Model

In [ ]:
class ViTModel(nn.Module):
    def __init__(self, patch_size=2, embed_dim=32, num_heads=4, num_layers=2, num_classes=10):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = 16

        # Project each 2x2 patch (4 pixel values) into an embedding
        self.patch_projection = nn.Linear(4, embed_dim)

        # Positional embedding
        self.position_embedding = nn.Parameter(torch.randn(1, 16, embed_dim) * 0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=64,
            dropout=0.1,
            batch_first=True,
            activation="gelu"
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, num_classes)
        )

    def create_patches(self, x):
        # Input shape: [B, 1, 8, 8]
        batch_size = x.shape[0]
        x = x.unfold(2, 2, 2).unfold(3, 2, 2)
        # Shape: [B, 1, 4, 4, 2, 2]
        x = x.contiguous().view(batch_size, 1, 16, 4)
        # Shape: [B, 1, 16, 4]
        x = x.squeeze(1)
        # Shape: [B, 16, 4]
        return x

    def forward(self, x):
        x = self.create_patches(x)
        x = self.patch_projection(x)
        x = x + self.position_embedding
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.classifier(x)
        return x

4. Test Model Input and Output

In [ ]:
cnn = CNNModel()
vit = ViTModel()

# Batch of 4 sample grayscale images, each 8x8
sample = torch.randn(4, 1, 8, 8)

cnn_output = cnn(sample)
vit_output = vit(sample)
patches = vit.create_patches(sample)

print("Input shape:", sample.shape)
print("CNN output:", cnn_output.shape)
print("ViT patches:", patches.shape)
print("ViT output:", vit_output.shape)

Input shape: torch.Size([4, 1, 8, 8])
CNN output: torch.Size([4, 10])
ViT patches: torch.Size([4, 16, 4])
ViT output: torch.Size([4, 10])
